In [1]:
import os

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/minhkhai0402/data_science_proj_test.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "minhkhai0402"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "dd58a3efc6d9346727fda8ab8ca2e3ec9e72c48d"

In [2]:
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\data_science_proj_test\\research'

In [3]:
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\data_science_proj_test'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [11]:
from src.data_science.constants import *
from src.data_science.utils.common import create_directories, read_yaml, save_json

class ConfigManager:
    def __init__(self, 
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN
        
        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema.name,
            mlflow_uri="https://dagshub.com/minhkhai0402/data_science_proj_test.mlflow"
        )

        return model_evaluation_config

In [14]:
import os, pandas as pd, numpy as np
import joblib
import mlflow, mlflow.sklearn
from urllib.parse import urlparse
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
    
    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae  = mean_absolute_error(actual, pred)
        r2   = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)
        
        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]
        
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            
            predicted_qualities = model.predict(test_x)

            (self.rmse, self.mae, self.r2_score) = self.eval_metrics(test_y, predicted_qualities)
            score = {"rmse": self.rmse, "mae": self.mae, "r2_score": self.r2_score}
            save_json(path=Path(self.config.metric_file_name), data=score)
            
            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", self.rmse)
            mlflow.log_metric("mae", self.mae)
            mlflow.log_metric("r2_score", self.r2_score)

            
            if tracking_url_type_store != 'file':
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetWineModel")
            else:
                mlflow.sklearn.log_model(model, "model")

In [15]:
try:
    config = ConfigManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-03-22 18:14:28,756: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-22 18:14:28,758: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-22 18:14:28,761: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-22 18:14:28,762: INFO: common: created directory at: artifacts]
[2026-03-22 18:14:28,763: INFO: common: created directory at: artifacts/model_evaluation]
[2026-03-22 18:14:29,230: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]


2026/03/22 18:14:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/22 18:14:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'ElasticnetWineModel'.
2026/03/22 18:14:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetWineModel, version 1
Created version '1' of model 'ElasticnetWineModel'.


🏃 View run fun-whale-734 at: https://dagshub.com/minhkhai0402/data_science_proj_test.mlflow/#/experiments/0/runs/73d7a2de5e194ef79807e7d0a1840093
🧪 View experiment at: https://dagshub.com/minhkhai0402/data_science_proj_test.mlflow/#/experiments/0
